# 06 — Inference on a Spatial Test Block

Feeds a **single held-out test block** from the materialized spatial split
(`/Volumes/T7/.../data/spatial_split/`) through a trained model and compares the
prediction with the CDL ground truth.

The split (block=1024px, 70/15/15, seed=42) was materialized by
`scratchpad/materialize_split.py`: each block is a folder with `s2.tif`
(23 dates × 10 bands = 230 channels) + `cdl.tif`, under `train/ val/ test/`.
This notebook picks one **test** block — spatially disjoint from training —
so the demo shows generalization to unseen ground.

In [14]:
# Best model directory across scenarios
import os

best_model_root_dir = "/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs"
best_model_dir = {
    "single_date":
        {
            "segformer": "exp_single_date_segformer_20260630-041347",
            "dlv3plus_cbam": "exp_single_date_deeplabv3plus_cbam_20260630-042801"
        },
    "mt_ndvi":
        {
            "segformer": "exp_mt_base_segformer_20260630-044523",
            "dlv3plus_cbam": "exp_mt_base_deeplabv3plus_cbam_20260630-050013"
        },
    "gsi":
        {
            "segformer": "exp_gsi_segformer_20260630-061835",
            "dlv3plus_cbam": "exp_gsi_deeplabv3plus_cbam_20260630-063939"
        },
    "rf":
        {
            "segformer": "exp_rf_segformer_20260630-055524",
            "dlv3plus_cbam": "exp_rf_deeplabv3plus_cbam_20260630-060808"
        }
}

for scenario, models in best_model_dir.items():
    print(f"\n=== {scenario} ===")
    for model_name, run_dir in models.items():
        full_path = os.path.join(best_model_root_dir, run_dir)
        print(f"{model_name}:")
        print(full_path)


=== single_date ===
segformer:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_single_date_segformer_20260630-041347
dlv3plus_cbam:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_single_date_deeplabv3plus_cbam_20260630-042801

=== mt_ndvi ===
segformer:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_mt_base_segformer_20260630-044523
dlv3plus_cbam:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_mt_base_deeplabv3plus_cbam_20260630-050013

=== gsi ===
segformer:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_gsi_segformer_20260630-061835
dlv3plus_cbam:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gm

In [15]:
# Register this repo as `cropmap_pipeline` regardless of checkout dir name.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'cropmap_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from cropmap_pipeline import config as C
print('Repo:', REPO, '| classes:', C.NUM_CLASSES)

Repo: /Users/dikaizm/Documents/PROGRAMMING/ml-ai/research-crop-mapping-thesis/research-crop-mapping-geoai/cropmap-remote-sensing-exps | classes: 9


In [16]:
import json, glob
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

from cropmap_pipeline.stages.training.train_segmentation import build_model, evaluate_test_set
from cropmap_pipeline.stages.training.normalization import _per_channel_percentiles

SPLIT_DIR = Path('/Volumes/T7/research-crop-mapping-geoai/data/spatial_split')
SCENARIO  = 'gsi'          # single_date | mt_ndvi | gsi | rf
ARCH      = 'segformer'    # must match the checkpoint
THRESH    = 0.5
PATCH     = C.PATCH_SIZE
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
manifest = json.loads((SPLIT_DIR / 'blocks_manifest.json').read_text())
test_blocks = [b for b in manifest['blocks'] if b['split'] == 'test']
print('test blocks:', [Path(b['s2']).parent.name for b in test_blocks])

test blocks: ['block_r0_c1', 'block_r2_c1', 'block_r2_c5', 'block_r4_c0', 'block_r4_c5', 'block_r5_c3', 'block_r5_c4']


## 1. Load one test block (S2 stack + CDL)

In [17]:
BLK = test_blocks[0]                       # pick the first test block
blk_dir = SPLIT_DIR / Path(BLK['s2']).parent
with rasterio.open(SPLIT_DIR / BLK['s2']) as s:
    s2 = s.read().astype(np.float32)       # (230, H, W)
    band_names = list(s.descriptions)
with rasterio.open(SPLIT_DIR / BLK['cdl']) as s:
    cdl = s.read(1).astype(np.int32)
gt = C.REMAP_LUT[np.clip(cdl, 0, 255)].astype(np.int64)   # 0=bg, 1..8 crops
print(f'block {blk_dir.name}: S2 {s2.shape} | {len(band_names)} channels | crops present:',
      sorted(set(np.unique(gt)) - {0}))

block block_r0_c1: S2 (230, 1024, 1024) | 230 channels | crops present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## 2. Block RGB + ground truth

In [18]:
def bidx(name): return band_names.index(name)
def stretch(a):
    a = a.copy(); a[a == C.S2_NODATA] = np.nan
    lo, hi = np.nanpercentile(a, [2, 98])
    return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)

mid = band_names[len(band_names)//2].split('_')[1]   # a mid-year date
rgb = np.dstack([stretch(s2[bidx(f'{b}_{mid}')]) for b in ('B4','B3','B2')])
pal = plt.cm.tab10(np.linspace(0,1,len(C.CDL_CLASS_NAMES)))
cmap = ListedColormap([(.9,.9,.9,1)]+[tuple(c) for c in pal])
norm = BoundaryNorm(np.arange(-.5, C.NUM_CLASSES+.5), C.NUM_CLASSES)
fig,ax = plt.subplots(1,2,figsize=(12,6))
ax[0].imshow(rgb); ax[0].set_title(f'{blk_dir.name} RGB ({mid})'); ax[0].axis('off')
ax[1].imshow(gt, cmap=cmap, norm=norm, interpolation='nearest'); ax[1].set_title('CDL ground truth'); ax[1].axis('off')
ax[1].legend(handles=[Patch(facecolor='0.9',label='bg')]+[Patch(facecolor=pal[i],label=n) for i,n in enumerate(C.CDL_CLASS_NAMES.values())],
             bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Select the scenario's channels from the block

Selection channel names (`band_YYYYMMDD`) match the block's band descriptions directly, so map by name. (single_date/mt_ndvi would pick dates instead.)

In [19]:
if SCENARIO in ('gsi', 'rf'):
    sel = json.loads((SPLIT_DIR.parent / f'select_{SCENARIO}_direct_s{THRESH:g}.json').read_text())
    want = sel['union_channels']
else:
    want = band_names   # baselines: all channels (date sub-selection omitted in this demo)
name_to_i = {n: i for i, n in enumerate(band_names)}
chan_idx = [name_to_i[n] for n in want if n in name_to_i]
print(f'{SCENARIO}: {len(chan_idx)} / {len(band_names)} channels selected')

gsi: 155 / 230 channels selected


## 4. Load both architectures + tiled inference

Load the best `SegFormer` and `DeepLabV3+CBAM` checkpoints for this scenario
(`best_model_dir[SCENARIO]`), normalize the block with the training per-band stats,
and predict tile-by-tile (256 px). Architecture/channels come from each checkpoint.

In [ ]:
# Each checkpoint stores the EXACT channels + order it trained on (band_names).
# Use those (not the current selection JSON, which may differ) so inputs match the model.
d = np.load(SPLIT_DIR / 'norm_stats_percentile.npz'); lo_b, hi_b = d['lo'], d['hi']
name_to_i = {n: i for i, n in enumerate(band_names)}          # block stack index by channel name
band_of   = lambda nm: C.S2_BAND_NAMES.index(nm.split('_')[0])

def load_model(ckpt_path):
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    arch  = ck.get('architecture', ARCH); inch = ck.get('in_channels'); ncls = ck.get('num_classes', C.NUM_CLASSES)
    m = build_model(arch, inch, ncls).to(DEVICE); m.load_state_dict(state); m.eval()
    return m, arch, ck

def infer(model, chan_idx, lo, hi):
    den = np.maximum(hi - lo, 1.0); n = len(chan_idx)
    Hh, Ww = s2.shape[1], s2.shape[2]; pred = np.zeros((Hh, Ww), np.uint8)
    with torch.no_grad():
        for y in range(0, Hh, PATCH):
            for x in range(0, Ww, PATCH):
                ph, pw = min(PATCH, Hh-y), min(PATCH, Ww-x)
                tile = s2[np.ix_(chan_idx, range(y,y+ph), range(x,x+pw))].astype(np.float32)
                tile[tile == C.S2_NODATA] = 0.0; tile[~np.isfinite(tile)] = 0.0
                tile = np.clip((tile - lo[:,None,None]) / den[:,None,None], 0, 1)
                pad = np.zeros((n, PATCH, PATCH), np.float32); pad[:, :ph, :pw] = tile
                out = model(torch.from_numpy(pad).unsqueeze(0).to(DEVICE)).argmax(1)[0].cpu().numpy()
                pred[y:y+ph, x:x+pw] = out[:ph, :pw]
    return pred

PREDS = {}
for label, run_dir in best_model_dir[SCENARIO].items():
    ckpt = os.path.join(best_model_root_dir, run_dir, 'best_model.pth')
    if not os.path.exists(ckpt):
        print(f'  ! missing checkpoint for {label}: {ckpt}'); continue
    model, arch, ck = load_model(ckpt)
    names = ck['band_names']                                   # training channels, in order
    miss = [nm for nm in names if nm not in name_to_i]
    assert not miss, f'{label}: {len(miss)} channels absent from block, e.g. {miss[:3]}'
    ci = [name_to_i[nm] for nm in names]
    lo = np.array([lo_b[band_of(nm)] for nm in names], np.float32)
    hi = np.array([hi_b[band_of(nm)] for nm in names], np.float32)
    PREDS[label] = infer(model, ci, lo, hi)
    print(f"  {label:16s} → {arch} ({ck['in_channels']}ch, epoch {ck.get('epoch','?')}, "
          f"val mIoU {ck.get('best_miou',float('nan')):.4f}) inferred")
assert PREDS, 'No checkpoints loaded — check best_model_root_dir / GDrive mount.'

## 5. Visualize — ground truth vs both architectures

Test block predictions (spatially disjoint from training). Per-crop IoU compared
across SegFormer and DeepLabV3+CBAM.

In [ ]:
def iou(g, p, k):
    i = ((g==k)&(p==k)).sum(); u = ((g==k)|(p==k)).sum(); return (i/u) if u else np.nan
labels = list(PREDS)
metrics = {}
for label, pred in PREDS.items():
    per = {C.CDL_CLASS_NAMES[c]: iou(gt, pred, k+1) for k, c in enumerate(C.KEEP_CLASSES)}
    metrics[label] = {'mIoU': float(np.nanmean([v for v in per.values() if not np.isnan(v)])),
                      'OA': float((gt == pred).mean()), 'per_crop': per}
    print(f"{label:16s} mIoU {metrics[label]['mIoU']:.4f} | OA {metrics[label]['OA']:.4f}")

# ── maps: row1 = GT + each prediction; row2 = RGB + each error map ──
ncol = 1 + len(labels)
fig, ax = plt.subplots(2, ncol, figsize=(6*ncol, 12))
ax[0,0].imshow(gt, cmap=cmap, norm=norm, interpolation='nearest'); ax[0,0].set_title('Ground Truth (CDL)')
ax[1,0].imshow(rgb); ax[1,0].set_title(f'RGB ({mid})')
for j, label in enumerate(labels, 1):
    ax[0,j].imshow(PREDS[label], cmap=cmap, norm=norm, interpolation='nearest')
    ax[0,j].set_title(f"{label}\nmIoU {metrics[label]['mIoU']:.3f} · OA {metrics[label]['OA']:.3f}")
    ax[1,j].imshow((gt != PREDS[label]).astype(int), cmap='Reds', interpolation='nearest')
    ax[1,j].set_title(f'{label} — errors')
for a in ax.flat: a.axis('off')
ax[0,0].legend(handles=[Patch(facecolor='0.9',label='bg')]+[Patch(facecolor=pal[i],label=nm) for i,nm in enumerate(C.CDL_CLASS_NAMES.values())],
               bbox_to_anchor=(0,0), loc='upper right', fontsize=7)
plt.suptitle(f'Test block {blk_dir.name} — {SCENARIO}', fontsize=15); plt.tight_layout(); plt.show()

# ── per-crop IoU grouped bar ──
crops = list(C.CDL_CLASS_NAMES.values()); x = np.arange(len(crops)); w = 0.8/len(labels)
fig, axb = plt.subplots(figsize=(12, 5))
for k, label in enumerate(labels):
    axb.bar(x + k*w, [metrics[label]['per_crop'][c] for c in crops], w, label=label)
axb.set_xticks(x + w*(len(labels)-1)/2); axb.set_xticklabels(crops, rotation=30, ha='right')
axb.set_ylabel('IoU'); axb.set_ylim(0, 1); axb.legend()
axb.set_title(f'Per-crop IoU — {SCENARIO} @ test block {blk_dir.name}')
plt.tight_layout(); plt.show()